# Model Performansini Artirmak Icin Yeni Ozellik Muhendisligi

## Gorev Tanimi
Musteri davranislarini daha iyi tahmin etmek icin mevcut ozelliklere ek olarak yeni ozellikler gelistirilmis ve XGBoost modeli uzerinde test edilerek performans karsilastirilmistir.

## Gelistirilen Ozellik Gruplari
| Grup | Ozellikler | Mantik |
|------|-----------|--------|
| 1. Sayfa Basina Ort. Zaman | avg_product_time, avg_admin_time, avg_info_time | Sayfa basina harcanan zaman satin alma niyetini gosterir |
| 2. Oturum Kompozisyonu | total_pages, product_pages_ratio, info_pages_ratio | Ziyaretcinin ne kadar odakli alisveris yaptigini olcer |
| 3. Etkilesim Kalitesi | bounce_exit_gap, page_value_per_prod, session_value_score | Ziyaretin kalitesini ve satin alma potansiyelini olcer |
| 4. Capraz Ozellikler | prod_dur_x_page_val, weekend_x_special | Birden fazla sinyalin birlikte etkisini yakalar |
| 5. Mevsimsellik | is_holiday_season, is_q1_shopping, is_summer | Alisveris davranisinin mevsime gore degisimini modeller |
| 6. Ziyaretci Segmenti | is_returning, returning_x_page_value | Geri donen musterilerin farkli satin alma patikasiyla etkilesimi |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, recall_score, f1_score,
                              precision_score, roc_auc_score, confusion_matrix)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
plt.rcParams['figure.dpi'] = 100
print('Kutuphaneler yuklendi.')

## 1. Veri Yukleme

In [ ]:
def load():
    try:
        data = pd.read_csv('/kaggle/input/datasets/imakash3011/online-shoppers-purchasing-intention-dataset/online_shoppers_intention.csv')
        print('Kaggle ortamindan yuklendi.')
    except FileNotFoundError:
        data = pd.read_csv('../online_shoppers_intention.csv')
        print('Lokal dosyadan yuklendi.')
    return data

df = load()
print(f'Veri boyutu: {df.shape}')
print(f'\nHedef degisken (Revenue) dagilimi:')
print(df['Revenue'].value_counts())
print(f'\nEksik deger: {df.isnull().sum().sum()}')

## 2. Baseline Pipeline (Orijinal Ozellikler)
Mevcut notebookta uygulanan pipeline'in tam kopyasi. Karsilastirma referansi olarak kullanilacak.

In [ ]:
def build_baseline(df_raw):
    """Orijinal e-commerce-predict-system.ipynb ile ayni pipeline."""
    df = df_raw.copy()

    # Log donusumu (carpik surekli degiskenler)
    skew_cols = ['Administrative_Duration', 'Informational_Duration',
                 'ProductRelated_Duration', 'PageValues']
    for col in skew_cols:
        df[col] = np.log1p(df[col])

    # Label encoding (bool sutunlar)
    le = LabelEncoder()
    bool_cols = [col for col in df.columns if df[col].dtype == bool]
    for col in bool_cols:
        df[col] = le.fit_transform(df[col])

    # Hedef degiskeni ayir
    y = df['Revenue'].astype(int)
    X = df.drop('Revenue', axis=1)

    # One-hot encoding
    X = pd.get_dummies(X, columns=['Month', 'VisitorType'], drop_first=True)

    return X, y


X_baseline, y = build_baseline(df)
print(f'Baseline ozellik sayisi: {X_baseline.shape[1]}')
print(f'Ozellik listesi: {list(X_baseline.columns)}')

In [ ]:
# RandomizedSearchCV ile belirlenmis en iyi parametreler
BEST_PARAMS = dict(
    n_estimators=1500,
    max_depth=6,
    learning_rate=0.1,
    subsample=1.0,
    colsample_bytree=0.7,
    gamma=2,
    reg_lambda=2,
    min_child_weight=1,
    eval_metric='logloss',
    random_state=42
)

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'F1':        f1_score(y_test, y_pred),
        'AUC-ROC':   roc_auc_score(y_test, y_prob)
    }


X_train_b, X_test_b, y_train, y_test = train_test_split(
    X_baseline, y, test_size=0.2, random_state=42, stratify=y
)

xgb_baseline = XGBClassifier(**BEST_PARAMS)
xgb_baseline.fit(X_train_b, y_train)

baseline_metrics = evaluate(xgb_baseline, X_test_b, y_test)

print('=== BASELINE XGBoost Sonuclari ===')
for k, v in baseline_metrics.items():
    print(f'  {k:<12}: {v:.4f}')
print(f'\nConfusion Matrix:\n{confusion_matrix(y_test, xgb_baseline.predict(X_test_b))}')

## 3. Yeni Ozellik Muhendisligi

Ham veriden, musteri satin alma davranisini aciklamaya yardimci olacak 16 yeni ozellik turetilmistir.

In [ ]:
def engineer_features(df_raw):
    """
    Ham veri uzerinde musteri davranisini modellemek icin
    16 yeni ozellik turetir.
    """
    df = df_raw.copy()

    # -------------------------------------------------------
    # GRUP 1: Sayfa Basina Ortalama Zaman (Engagement Derinligi)
    # Her sayfa turunde ortalama gecirilen zaman, ziyaretcinin
    # o icerikle ne kadar ilgilendigini gosterir.
    # -------------------------------------------------------
    df['avg_product_time'] = df['ProductRelated_Duration'] / (df['ProductRelated'] + 1)
    df['avg_admin_time']   = df['Administrative_Duration'] / (df['Administrative'] + 1)
    df['avg_info_time']    = df['Informational_Duration']  / (df['Informational'] + 1)

    # -------------------------------------------------------
    # GRUP 2: Oturum Kompozisyon Ozellikleri
    # Ziyaretcinin hangi tur sayfalari ne kadar ziyaret ettigini
    # ve urun sayfasina ne kadar odaklandigini olcer.
    # -------------------------------------------------------
    df['total_pages']         = (df['Administrative'] +
                                  df['Informational'] +
                                  df['ProductRelated'])
    df['total_duration']      = (df['Administrative_Duration'] +
                                  df['Informational_Duration'] +
                                  df['ProductRelated_Duration'])
    df['product_pages_ratio'] = df['ProductRelated'] / (df['total_pages'] + 1)
    df['info_pages_ratio']    = df['Informational']  / (df['total_pages'] + 1)

    # -------------------------------------------------------
    # GRUP 3: Etkilesim Kalitesi Gostergeleri
    # Bounce/Exit oranlari ile sayfa degerlerini birlestirerek
    # ziyaretin 'kalitesini' ozetler.
    # -------------------------------------------------------
    # ExitRate > BounceRate: kullanici biraz gezdikten sonra cikiyor (katilim var)
    df['bounce_exit_gap']    = df['ExitRates'] - df['BounceRates']
    # Urun sayfasi basina dusun ortalama sayfa degeri
    df['page_value_per_prod'] = df['PageValues'] / (df['ProductRelated'] + 1)
    # Dusuk bounce + yuksek sayfa degeri: guclu satin alma niyeti sinyali
    df['session_value_score'] = df['PageValues'] / (df['BounceRates'] + 0.01)

    # -------------------------------------------------------
    # GRUP 4: Mevsimsellik Ozellikleri
    # Alisveris davranisinin yilin belirli donemlerinde sistematik
    # olarak degistigi bilinmektedir.
    # -------------------------------------------------------
    df['is_holiday_season'] = df['Month'].isin(['Nov', 'Dec']).astype(int)
    df['is_q1_shopping']    = df['Month'].isin(['Feb', 'Mar']).astype(int)
    df['is_summer']         = df['Month'].isin(['Jun', 'Jul', 'Aug']).astype(int)

    # -------------------------------------------------------
    # GRUP 5: Ziyaretci Segment Etkilesim Ozellikleri
    # Geri donen musterilerin satin alma davranisi yeni
    # ziyaretcilerden sistematik olarak farklidir.
    # -------------------------------------------------------
    df['is_returning']           = (df['VisitorType'] == 'Returning_Visitor').astype(int)
    # Geri donen + yuksek sayfa degeri: en guclu satin alma potansiyeli
    df['returning_x_page_value'] = df['is_returning'] * df['PageValues']
    # Geri donen + urun odakli gezinme birlikteligi
    df['returning_x_prod_ratio'] = df['is_returning'] * df['product_pages_ratio']

    return df


df_enh = engineer_features(df)
new_cols = [c for c in df_enh.columns if c not in df.columns]
print(f'Eklenen yeni ozellik sayisi: {len(new_cols)}')
print('\nYeni ozellikler:')
for i, col in enumerate(new_cols, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
# Yeni ozelliklerin hedef degiskenle korelasyonu
df_corr = df_enh.copy()
df_corr['Revenue_int'] = df_corr['Revenue'].astype(int)
corr_vals = df_corr[new_cols + ['Revenue_int']].corr()['Revenue_int'].drop('Revenue_int').sort_values(key=abs, ascending=False)

colors = ['#4CAF50' if v > 0 else '#F44336' for v in corr_vals.values]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(corr_vals)), corr_vals.values, color=colors, alpha=0.85)
ax.set_yticks(range(len(corr_vals)))
ax.set_yticklabels(corr_vals.index, fontsize=10)
ax.set_xlabel('Korelasyon Katsayisi (Revenue ile)', fontsize=11)
ax.set_title('Yeni Ozelliklerin Hedef Degiskenle Korelasyonu\n(Yesil: Pozitif, Kirmizi: Negatif)', fontsize=13, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
for i, v in enumerate(corr_vals.values):
    ax.text(v + (0.002 if v >= 0 else -0.002), i, f'{v:.3f}',
            va='center', ha='left' if v >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.savefig('feature_correlations.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Gelismis Pipeline (Yeni Ozelliklerle)

In [ ]:
def build_enhanced(df_raw):
    """Orijinal pipeline + yeni ozellikler."""
    df = engineer_features(df_raw)

    # Log donusumu: orijinal + yeni surekli ozellikler
    log_cols = [
        # Orijinal carpik sutunlar
        'Administrative_Duration', 'Informational_Duration',
        'ProductRelated_Duration', 'PageValues',
        # Yeni ozellikler - sag carpik dagilimi duz
        'avg_product_time', 'avg_admin_time', 'avg_info_time',
        'total_duration', 'page_value_per_prod',
        'session_value_score', 'returning_x_page_value'
    ]
    for col in log_cols:
        df[col] = np.log1p(df[col])

    # Label encoding (bool sutunlar)
    le = LabelEncoder()
    bool_cols = [c for c in df.columns if df[c].dtype == bool]
    for col in bool_cols:
        df[col] = le.fit_transform(df[col])

    # Log donusumu sonrasi capraz ozellikler
    # (log donusum sonrasi daha dengeli iki sinyal)
    df['prod_dur_x_page_val'] = df['ProductRelated_Duration'] * df['PageValues']
    df['weekend_x_special']   = df['Weekend'] * df['SpecialDay']

    # Hedef degisken
    y = df['Revenue'].astype(int)
    X = df.drop('Revenue', axis=1)

    # One-hot encoding
    X = pd.get_dummies(X, columns=['Month', 'VisitorType'], drop_first=True)

    return X, y


X_enhanced, y_enh = build_enhanced(df)

added = X_enhanced.shape[1] - X_baseline.shape[1]
print(f'Baseline ozellik sayisi   : {X_baseline.shape[1]}')
print(f'Eklenen ozellik sayisi    : {added}')
print(f'Toplam ozellik sayisi     : {X_enhanced.shape[1]}')

## 5. Gelismis Model Egitimi ve Degerlendirmesi

In [ ]:
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_enhanced, y_enh, test_size=0.2, random_state=42, stratify=y_enh
)

xgb_enhanced = XGBClassifier(**BEST_PARAMS)
xgb_enhanced.fit(X_train_e, y_train_e)

enhanced_metrics = evaluate(xgb_enhanced, X_test_e, y_test_e)

print('=== GELISMIS MODEL Sonuclari ===')
for k, v in enhanced_metrics.items():
    print(f'  {k:<12}: {v:.4f}')
print(f'\nConfusion Matrix:\n{confusion_matrix(y_test_e, xgb_enhanced.predict(X_test_e))}')

## 6. Performans Karsilastirmasi

In [ ]:
metrics_list = list(baseline_metrics.keys())
b_values     = [baseline_metrics[m] for m in metrics_list]
e_values     = [enhanced_metrics[m] for m in metrics_list]
delta        = [e - b for e, b in zip(e_values, b_values)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Yan yana cubuk grafigi ---
x     = np.arange(len(metrics_list))
width = 0.35
bars_b = axes[0].bar(x - width/2, b_values, width, label='Baseline',      color='#2196F3', alpha=0.85)
bars_e = axes[0].bar(x + width/2, e_values, width, label='Gelismis Model', color='#FF6B35', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_list, fontsize=10)
axes[0].set_ylim(0, 1.08)
axes[0].set_title('Baseline vs Gelismis Model\nPerformans Karsilastirmasi', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylabel('Skor')
for bar in bars_b:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars_e:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.3f}', ha='center', va='bottom', fontsize=8)

# --- Delta grafigi ---
colors_delta = ['#4CAF50' if d > 0 else '#F44336' if d < 0 else '#9E9E9E' for d in delta]
axes[1].bar(metrics_list, delta, color=colors_delta, alpha=0.85)
axes[1].axhline(y=0, color='black', linewidth=0.8)
axes[1].set_title('Metrik Degisim Miktarlari\n(Gelismis - Baseline)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Delta')
axes[1].grid(axis='y', alpha=0.3)
for i, d in enumerate(delta):
    sign = '+' if d >= 0 else ''
    axes[1].text(i, d + (0.0005 if d >= 0 else -0.001),
                 f'{sign}{d:.4f}', ha='center',
                 va='bottom' if d >= 0 else 'top',
                 fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('performance_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Ozellik Onem Analizi

In [ ]:
# Gelismis modeldeki tum ozellik onemleri
feat_imp = pd.DataFrame({
    'feature':    X_enhanced.columns.tolist(),
    'importance': xgb_enhanced.feature_importances_
}).sort_values('importance', ascending=False).head(25)

new_feature_names = [c for c in X_enhanced.columns if c not in X_baseline.columns]
feat_imp['is_new'] = feat_imp['feature'].isin(new_feature_names)

colors_imp = ['#FF6B35' if is_new else '#2196F3' for is_new in feat_imp['is_new']]

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(range(len(feat_imp)), feat_imp['importance'], color=colors_imp, alpha=0.85)
ax.set_yticks(range(len(feat_imp)))
ax.set_yticklabels(feat_imp['feature'], fontsize=9)
ax.set_xlabel('Ozellik Onemi (XGBoost Gain)', fontsize=11)
ax.set_title('Top 25 Ozellik Onemi\n(Turuncu: Yeni, Mavi: Mevcut)', fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#FF6B35', alpha=0.85, label='Yeni Ozellikler'),
    Patch(facecolor='#2196F3', alpha=0.85, label='Mevcut Ozellikler')
]
ax.legend(handles=legend_elements, fontsize=10)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n=== Yeni Ozelliklerin Siralamasi (Top 25 icinde) ===')
for _, row in feat_imp[feat_imp['is_new']].iterrows():
    rank = feat_imp['feature'].tolist().index(row['feature']) + 1
    print(f'  #{rank:2d}  {row["feature"]:<30}: {row["importance"]:.4f}')

## 8. Capraz Dogrulama (5-Fold CV) Karsilastirmasi

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring_map = {'F1': 'f1', 'Recall': 'recall', 'Precision': 'precision', 'AUC-ROC': 'roc_auc'}

print('=== 5-FOLD CAPRAZ DOGRULAMA KARSILASTIRMASI ===\n')
cv_summary = {}

for metric_name, scoring in scoring_map.items():
    cv_b = cross_val_score(XGBClassifier(**BEST_PARAMS), X_baseline, y,     cv=cv, scoring=scoring)
    cv_e = cross_val_score(XGBClassifier(**BEST_PARAMS), X_enhanced, y_enh, cv=cv, scoring=scoring)
    cv_summary[metric_name] = {
        'baseline':  (cv_b.mean(), cv_b.std()),
        'enhanced':  (cv_e.mean(), cv_e.std()),
        'delta':     cv_e.mean() - cv_b.mean()
    }
    sign = '+' if cv_summary[metric_name]['delta'] >= 0 else ''
    print(f'{metric_name}:')
    print(f'  Baseline    : {cv_b.mean():.4f} +/- {cv_b.std():.4f}')
    print(f'  Gelismis    : {cv_e.mean():.4f} +/- {cv_e.std():.4f}')
    print(f'  Degisim     : {sign}{cv_e.mean() - cv_b.mean():.4f}\n')

## 9. Ozet Rapor

In [ ]:
print('=' * 65)
print('      OZET: YENI OZELLIK MUHENDISLIGI SONUCLARI')
print('=' * 65)

print(f'\nOzellik Bilgileri:')
print(f'  Baseline ozellik sayisi   : {X_baseline.shape[1]}')
print(f'  Yeni eklenen ozellik sayisi: {len(new_feature_names)}')
print(f'  Toplam ozellik sayisi     : {X_enhanced.shape[1]}')

print(f'\nTest Seti Performans Karsilastirmasi:')
print(f'{"Metrik":<14} {"Baseline":>10} {"Gelismis":>12} {"Delta":>10} {"Durum":>8}')
print('-' * 58)
for metric in baseline_metrics:
    b = baseline_metrics[metric]
    e = enhanced_metrics[metric]
    d = e - b
    sign = '+' if d >= 0 else ''
    status = 'IYI' if d > 0.001 else ('KOTU' if d < -0.001 else 'AYNI')
    print(f'{metric:<14} {b:>10.4f} {e:>12.4f} {sign+f"{d:.4f}":>10} {status:>8}')

print('\nGelistirilen Ozellik Gruplari ve Ozellikleri:')
groups = [
    ('1. Sayfa Basina Ort. Zaman', ['avg_product_time', 'avg_admin_time', 'avg_info_time']),
    ('2. Oturum Kompozisyonu',     ['total_pages', 'total_duration', 'product_pages_ratio', 'info_pages_ratio']),
    ('3. Etkilesim Kalitesi',      ['bounce_exit_gap', 'page_value_per_prod', 'session_value_score']),
    ('4. Capraz Ozellikler',       ['prod_dur_x_page_val', 'weekend_x_special']),
    ('5. Mevsimsellik',            ['is_holiday_season', 'is_q1_shopping', 'is_summer']),
    ('6. Ziyaretci Segmenti',      ['is_returning', 'returning_x_page_value', 'returning_x_prod_ratio']),
]
for group_name, features in groups:
    print(f'  {group_name}: {", ".join(features)}')